# 04 · Continuous metrics — `foe.continuous`

**Hexkit surfaces:** `utility/automation_engine.py` (`run_comparison_suite`, `winsorize_series`),
`pages/user_level_analysis.py` (`run_business_case`; the page runs the same scipy tests as the
heuristic path inline), `pages/pre_test_analysis.py` (`fit_gamma`, `fit_negbin`).

| FOE function | R reference |
|---|---|
| `run_comparison_suite`, heuristic path: normality gate (D'Agostino K² on OLS residuals), Levene (median) gate, then ANOVA / Welch / Kruskal-Wallis / Mann-Whitney | `fBasics::dagoTest`, `car::leveneTest(center = median)`, `anova(lm)`, `oneway.test(var.equal = FALSE)`, `kruskal.test`, `wilcox.test` (exact/asymptotic chosen with scipy's rule) |
| heuristic path, count-KPI gate → Negative Binomial LRT | `MASS::glm.nb` (full vs intercept-only), `pchisq` |
| gamma path, per transaction: Gamma LRT + Bonferroni post-hoc | exact Gamma MLE (score equation `log k − ψ(k) = log x̄ − mean(log x)` solved with `uniroot`), `pchisq` |
| gamma path, per visitor: two-part hurdle LRT | Bernoulli log-likelihood + exact Gamma MLE on the positives |
| `fit_gamma`, `fit_negbin` | exact Gamma MLE and `MASS::fitdistr(x, "gamma")`; `MASS::fitdistr(x, "negative binomial")` |
| `winsorize_series` | `mean ± k·sd`, `quantile(type = 7)` |
| `run_business_case` (delta method) | `msm::deltamethod` |

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd

from foe.core.models import (AlternativeHypothesis, AnalysisUnit, ContinuousApproach,
                             ContinuousMetricConfig)
from foe.continuous.operations import ContinuousMetricEngine
from validation_helpers import Checks

checks = Checks()
engine = ContinuousMetricEngine()
rng = np.random.default_rng(2026)


def make_df(groups):
    """groups: {label: array} -> long DataFrame with columns variant, value."""
    return pd.DataFrame([(g, float(v)) for g, arr in groups.items() for v in arr],
                        columns=["variant", "value"])

## 1. Heuristic decision tree

One dataset per branch. The R function below re-implements the same decision tree using R's
own tests, so the comparison covers the gates (`is_normal`, `is_homogeneous`), the chosen
test and its p-value.

In [2]:
datasets = {
    "normal, equal var (ANOVA)": make_df({
        "A": rng.normal(50, 10, 400), "B": rng.normal(51.5, 10, 400), "C": rng.normal(50.5, 10, 400)}),
    "normal, unequal var (Welch)": make_df({
        "A": rng.normal(50, 5, 400), "B": rng.normal(51, 12, 400), "C": rng.normal(50, 20, 400)}),
    "skewed, 2 groups (Mann-Whitney)": make_df({
        "A": rng.lognormal(3.5, 0.9, 500), "B": rng.lognormal(3.6, 0.9, 520)}),
    "skewed, 3 groups (Kruskal-Wallis)": make_df({
        "A": rng.lognormal(3.5, 0.9, 400), "B": rng.lognormal(3.6, 0.9, 400), "C": rng.lognormal(3.45, 0.9, 400)}),
    "small sample (<20 rows, non-parametric)": make_df({
        "A": rng.normal(10, 2, 8), "B": rng.normal(12, 2, 9)}),
}

heur_py = []
for name, df in datasets.items():
    res = engine.run_comparison_suite(df, ContinuousMetricConfig(
        kpi="value", group_col="variant", approach=ContinuousApproach.HEURISTIC,
        unit=AnalysisUnit.PER_TRANSACTION, control_label="A"))
    heur_py.append(dict(name=name, test=res.test_name, p_value=res.p_value,
                        is_normal=res.is_normal, is_homogeneous=res.is_homogeneous))
heur_py = pd.DataFrame(heur_py)
heur_py

,name,test,p_value,is_normal,is_homogeneous
0,"normal, equal var (ANOVA)",Standard ANOVA,0.009942,True,True
1,"normal, unequal var (Welch)",Kruskal-Wallis,0.977833,False,False
2,"skewed, 2 groups (Mann-Whitney)",Mann-Whitney U,0.009730,False,True
3,"skewed, 3 groups (Kruskal-Wallis)",Kruskal-Wallis,0.470354,False,False
4,"small sample (<20 rows, non-parametric)",Mann-Whitney U,0.113945,False,True


In [3]:
%%R
suppressPackageStartupMessages({library(fBasics); library(car); library(MASS); library(msm)})

heuristic_r <- function(df) {
  df$variant <- factor(df$variant)
  fit <- lm(value ~ variant, data = df)
  res <- residuals(fit)
  is_normal <- if (length(res) >= 20) dagoTest(res)@test$p.value[1] >= 0.05 else FALSE
  is_homog  <- leveneTest(value ~ variant, data = df, center = median)[1, "Pr(>F)"] >= 0.05
  k <- nlevels(df$variant)
  if (is_normal && is_homog) {
    test <- "Standard ANOVA"; p <- anova(fit)[1, "Pr(>F)"]
  } else if (is_normal) {
    test <- "Welch's ANOVA"; p <- oneway.test(value ~ variant, data = df, var.equal = FALSE)$p.value
  } else if (k > 2) {
    test <- "Kruskal-Wallis"; p <- kruskal.test(value ~ variant, data = df)$p.value
  } else {
    g <- split(df$value, df$variant)
    test <- "Mann-Whitney U"
    # scipy's method="auto": exact if either arm has <= 8 rows and there are no ties
    exact <- min(lengths(g)) <= 8 && !any(duplicated(df$value))
    p <- wilcox.test(g[[1]], g[[2]], exact = exact, correct = TRUE)$p.value
  }
  data.frame(test = test, p_value = p, is_normal = is_normal, is_homogeneous = is_homog)
}

In [4]:
heur_r = []
for name, df in datasets.items():
    %R -i df -o out out <- heuristic_r(df)
    heur_r.append(out.iloc[0].to_dict() | {"name": name})
heur_r = pd.DataFrame(heur_r)
heur_r

,test,p_value,is_normal,is_homogeneous,name
0,Standard ANOVA,0.009942,True,True,"normal, equal var (ANOVA)"
1,Kruskal-Wallis,0.977833,False,False,"normal, unequal var (Welch)"
2,Mann-Whitney U,0.009730,False,True,"skewed, 2 groups (Mann-Whitney)"
3,Kruskal-Wallis,0.470354,False,False,"skewed, 3 groups (Kruskal-Wallis)"
4,Mann-Whitney U,0.113945,False,True,"small sample (<20 rows, non-parametric)"


In [5]:
for i, row in heur_py.iterrows():
    r = heur_r.iloc[i]
    checks.add_equal(f"is_normal       {row['name']}", bool(row.is_normal), bool(r.is_normal))
    checks.add_equal(f"is_homogeneous  {row['name']}", bool(row.is_homogeneous), bool(r.is_homogeneous))
    checks.add_equal(f"test chosen     {row['name']}", row.test, r.test)
    checks.add(f"p-value         {row['name']}", row.p_value, r.p_value, atol=1e-10, rtol=1e-6)

## 2. Count KPI → Negative Binomial likelihood-ratio test

Integer, low-cardinality KPIs (items per order, tickets…) skip the decision tree and go to an
NB2 regression LRT. R: `glm.nb(y ~ variant)` vs `glm.nb(y ~ 1)`, dispersion α = 1/θ.

Note: with `unit = per_transaction` (the config default) FOE drops zero rows *before* the
test, so the NB is fit to the positive counts only. That is right for "items per order"
(an order always has ≥ 1 item) but not for a per-visitor count with real zeros; both units are
checked here and R applies the same zero filter.

In [6]:
def nb_draw(mean, alpha, n):
    size = 1 / alpha
    return rng.negative_binomial(size, size / (size + mean), n)

count_sets = {
    "counts, 2 groups": make_df({"A": nb_draw(2.0, 0.6, 600), "B": nb_draw(2.3, 0.6, 600)}),
    "counts, 3 groups": make_df({"A": nb_draw(3.0, 0.4, 500), "B": nb_draw(3.0, 0.4, 500),
                                 "C": nb_draw(3.4, 0.4, 500)}),
}
nb_py = []
for name, df in count_sets.items():
    for unit in [AnalysisUnit.PER_VISITOR, AnalysisUnit.PER_TRANSACTION]:
        res = engine.run_comparison_suite(df, ContinuousMetricConfig(
            kpi="value", group_col="variant", approach=ContinuousApproach.HEURISTIC,
            unit=unit, control_label="A"))
        nb_py.append(dict(name=f"{name} | {unit.value}", unit=unit.value, data=name,
                          test=res.test_name, p_value=res.p_value, alpha=res.dispersion_alpha))
nb_py = pd.DataFrame(nb_py)
nb_py

,name,unit,data,test,p_value,alpha
0,"counts, 2 groups | per_visitor",per_visitor,"counts, 2 groups",Negative Binomial Regression (LRT),0.059802,0.669783
1,"counts, 2 groups | per_transaction",per_transaction,"counts, 2 groups",Negative Binomial Regression (LRT),0.018272,0.174781
2,"counts, 3 groups | per_visitor",per_visitor,"counts, 3 groups",Negative Binomial Regression (LRT),0.145331,0.436171
3,"counts, 3 groups | per_transaction",per_transaction,"counts, 3 groups",Negative Binomial Regression (LRT),0.196138,0.194951


In [7]:
nb_r = []
for _, row in nb_py.iterrows():
    df, unit = count_sets[row.data], row.unit
    %R -i df -i unit -o out if (unit == "per_transaction") df <- df[df$value != 0, ]; df$variant <- factor(df$variant); m1 <- glm.nb(value ~ variant, data = df); m0 <- glm.nb(value ~ 1, data = df); lr <- 2 * (as.numeric(logLik(m1)) - as.numeric(logLik(m0))); out <- data.frame(p_value = pchisq(lr, df = nlevels(df$variant) - 1, lower.tail = FALSE), alpha = 1 / m1$theta)
    nb_r.append(out.iloc[0].to_dict())
nb_r = pd.DataFrame(nb_r)
nb_r

,p_value,alpha
0,0.059802,0.669783
1,0.018272,0.174771
2,0.145331,0.436170
3,0.196138,0.194950


In [8]:
for i, row in nb_py.iterrows():
    r = nb_r.iloc[i]
    checks.add_equal(f"test chosen     {row['name']}", row.test, "Negative Binomial Regression (LRT)")
    checks.add(f"NB LRT p-value  {row['name']}", row.p_value, r.p_value, atol=1e-6, rtol=1e-3)
    checks.add(f"NB dispersion   {row['name']}", row.alpha, r.alpha, rtol=1e-3)

## 3. Gamma family: per-transaction Gamma LRT and per-visitor hurdle LRT

In [9]:
gamma_tx = make_df({"A": rng.gamma(2.0, 30.0, 400), "B": rng.gamma(2.0, 34.0, 420),
                    "C": rng.gamma(2.4, 25.0, 410)})

# Per-visitor: ~4% of visitors buy; non-buyers are 0.
def rpv(n, p, k, theta):
    return np.where(rng.random(n) < p, rng.gamma(k, theta, n), 0.0)
gamma_pv = make_df({"A": rpv(6000, 0.040, 2.0, 30.0), "B": rpv(6000, 0.046, 2.0, 30.0),
                    "C": rpv(6000, 0.041, 2.0, 33.0)})

gamma_py = {}
for unit, df in [(AnalysisUnit.PER_TRANSACTION, gamma_tx), (AnalysisUnit.PER_VISITOR, gamma_pv)]:
    res = engine.run_comparison_suite(df, ContinuousMetricConfig(
        kpi="value", group_col="variant", approach=ContinuousApproach.GAMMA_GLM,
        unit=unit, control_label="A"))
    gamma_py[unit.value] = res
    print(unit.value, res.test_name, res.p_value,
          [(p.comparison, p.p_value) for p in (res.posthoc_results or [])])

per_transaction Gamma GLM (Likelihood Ratio Test) 0.0035731812775433272 [('B vs A', 0.0015848660567646055), ('C vs A', 0.11330641104297562)]
per_visitor Two-Part Hurdle (Likelihood Ratio Test) 9.476171296680168e-06 [('B vs A', 0.0001274477905853484), ('C vs A', 0.00038755471770326755)]


In [10]:
%%R
# Exact Gamma MLE: the shape solves log(k) - digamma(k) = log(mean(x)) - mean(log(x)),
# then scale = mean(x) / k. Used instead of a general-purpose optimiser because LRT p-values
# depend on small log-likelihood differences.
gamma_mle <- function(x) {
  s <- log(mean(x)) - mean(log(x))
  k <- uniroot(function(k) log(k) - digamma(k) - s, c(1e-8, 1e8), tol = 1e-14)$root
  theta <- mean(x) / k
  list(k = k, theta = theta, ll = sum(dgamma(x, shape = k, scale = theta, log = TRUE)))
}
gamma_ll <- function(x) gamma_mle(x[x > 0])$ll
unit_ll <- function(x, unit) {
  if (unit == "per_transaction") return(gamma_ll(x))
  n <- length(x); k <- sum(x > 0); p <- k / n
  bern <- if (p > 0 && p < 1) k * log(p) + (n - k) * log(1 - p) else 0
  bern + gamma_ll(x)
}
gamma_lrt <- function(df, unit, control = "A") {
  npar <- if (unit == "per_transaction") 2 else 3
  if (unit == "per_transaction") df <- df[df$value > 0, ]
  groups <- split(df$value, df$variant)
  lr <- 2 * (sum(sapply(groups, unit_ll, unit = unit)) - unit_ll(df$value, unit))
  global <- pchisq(max(lr, 0), df = npar * (length(groups) - 1), lower.tail = FALSE)
  others <- setdiff(names(groups), control)
  post <- sapply(others, function(v) {
    lr <- 2 * (unit_ll(groups[[control]], unit) + unit_ll(groups[[v]], unit) -
               unit_ll(c(groups[[control]], groups[[v]]), unit))
    pchisq(max(lr, 0), df = npar, lower.tail = FALSE)
  })
  data.frame(term = c("global", paste(others, "vs", control)), p_value = c(global, post))
}

In [11]:
for unit, df in [("per_transaction", gamma_tx), ("per_visitor", gamma_pv)]:
    %R -i df -i unit -o out out <- gamma_lrt(df, unit)
    res = gamma_py[unit]
    checks.add(f"Gamma LRT global p  {unit}", res.p_value, out.p_value.iloc[0], atol=1e-8, rtol=1e-3)
    for ph in (res.posthoc_results or []):
        r_p = out.set_index("term").p_value[ph.comparison]
        checks.add(f"Gamma post-hoc p    {unit} {ph.comparison}", ph.p_value, r_p, atol=1e-8, rtol=1e-3)
    display(out)

,term,p_value
,global,0.003574
B,B vs A,0.001585
C,C vs A,0.113309


,term,p_value
,global,0.000009
B,B vs A,0.000127
C,C vs A,0.000388


## 4. Distribution fits used by the pre-test planner

In [12]:
x_gamma = rng.gamma(1.8, 40.0, 2000)
k, theta, ll = ContinuousMetricEngine.fit_gamma(x_gamma)
x_counts = nb_draw(2.5, 0.7, 2000).astype(float)
nb_ll, nb_alpha, nb_res = ContinuousMetricEngine.fit_negbin(x_counts)
nb_mean = float(np.exp(nb_res.params[0]))

In [13]:
%%R -i x_gamma -i x_counts -o fits_r
g <- gamma_mle(x_gamma)
g_fd <- fitdistr(x_gamma, "gamma", start = list(shape = 1, rate = 1 / mean(x_gamma)), lower = c(1e-8, 1e-10))
nb <- fitdistr(x_counts, "negative binomial")
fits_r <- data.frame(k = g$k, theta = g$theta, ll = g$ll,
                     k_fitdistr = g_fd$estimate[["shape"]], theta_fitdistr = 1 / g_fd$estimate[["rate"]],
                     nb_mu = nb$estimate[["mu"]], nb_alpha = 1 / nb$estimate[["size"]], nb_ll = nb$loglik)
fits_r

       k

   theta

        ll

 k_fitdistr

 theta_fitdistr

    nb_mu

... (8 outputs left out)

 0.600637

     nb_ll


1

 -4086.221

In [14]:
f = fits_r.iloc[0]
checks.add("fit_gamma shape k", k, f.k, rtol=1e-4)
checks.add("fit_gamma shape k (vs MASS::fitdistr)", k, f.k_fitdistr, rtol=5e-3)
checks.add("fit_gamma scale theta", theta, f.theta, rtol=1e-4)
checks.add("fit_gamma log-likelihood", ll, f.ll, rtol=1e-8)
checks.add("fit_negbin mean", nb_mean, f.nb_mu, rtol=1e-4)
checks.add("fit_negbin alpha", nb_alpha, f.nb_alpha, rtol=1e-3)
checks.add("fit_negbin log-likelihood", nb_ll, f.nb_ll, rtol=1e-8)

True

## 5. Winsorizing

In [15]:
w = pd.Series(rng.lognormal(3.5, 1.0, 3000))
_, sd_lo, sd_hi = ContinuousMetricEngine.winsorize_series(w, "Standard Deviation", 3.0)
_, pc_lo, pc_hi = ContinuousMetricEngine.winsorize_series(w, "Percentile", 1.0)
w = w.to_numpy()

In [16]:
%%R -i w -o wins_r
wins_r <- c(mean(w) - 3 * sd(w), mean(w) + 3 * sd(w), quantile(w, c(0.01, 0.99), type = 7))

In [17]:
checks.add("winsorize SD lower", sd_lo, wins_r[0], rtol=1e-10)
checks.add("winsorize SD upper", sd_hi, wins_r[1], rtol=1e-10)
checks.add("winsorize P1 lower", pc_lo, wins_r[2], rtol=1e-10)
checks.add("winsorize P99 upper", pc_hi, wins_r[3], rtol=1e-10)

True

## 6. Business case (`run_business_case`)

The projected value per visitor is `rate · mean` per arm; its SE comes from the delta method.
Per visitor, rate = 1; per transaction, rate = orders / visitors with a binomial SE.

In [18]:
group_stats = {"A": {"mean": 61.2, "std": 48.0, "count": 1450},
               "B": {"mean": 63.9, "std": 50.5, "count": 1510}}
visitor_counts = {"A": 40_000, "B": 40_200}
daily_visitors, period = 5_000, 183
bc = {}
for unit in [AnalysisUnit.PER_VISITOR, AnalysisUnit.PER_TRANSACTION]:
    bc[unit.value] = engine.run_business_case(
        group_stats=group_stats, control_label="A", unit=unit, daily_visitors=daily_visitors,
        visitor_counts=visitor_counts, alternative=AlternativeHypothesis.TWO_SIDED,
        projection_period=period)[0]
m_a, s_a, n_a = 61.2, 48.0, 1450
m_b, s_b, n_b = 63.9, 50.5, 1510
v_a, v_b = 40_000, 40_200
pd.DataFrame(bc).loc[["point_estimate", "ci_low", "ci_high"]]

,per_visitor,per_transaction
point_estimate,2470500.0,166277.350746
ci_low,-776372.489658,-25320.583933
ci_high,5717372.489658,357875.285425


In [19]:
%%R -i m_a -i s_a -i n_a -i m_b -i s_b -i n_b -i v_a -i v_b -i daily_visitors -i period -o bc_r
k <- daily_visitors * period
z <- qnorm(0.975)
# per visitor: X = m_b - m_a
se_pv <- deltamethod(~ x1 - x2, c(m_b, m_a), diag(c(s_b^2 / n_b, s_a^2 / n_a)))
x_pv <- m_b - m_a
# per transaction: X = r_b * m_b - r_a * m_a, r = orders / visitors
r_a <- n_a / v_a; r_b <- n_b / v_b
mu <- c(r_b, m_b, r_a, m_a)
V <- diag(c(r_b * (1 - r_b) / v_b, s_b^2 / n_b, r_a * (1 - r_a) / v_a, s_a^2 / n_a))
se_pt <- deltamethod(~ x1 * x2 - x3 * x4, mu, V)
x_pt <- r_b * m_b - r_a * m_a
bc_r <- data.frame(unit = c("per_visitor", "per_transaction"),
                   point_estimate = c(x_pv, x_pt) * k,
                   ci_low = (c(x_pv, x_pt) - z * c(se_pv, se_pt)) * k,
                   ci_high = (c(x_pv, x_pt) + z * c(se_pv, se_pt)) * k)
bc_r

            unit

 point_estimate

     ci_low

   ci_high


1

     per_visitor

... (4 outputs left out)

 per_transaction

       166277.4

  -25320.58

  357875.3

In [20]:
for _, r in bc_r.iterrows():
    for key in ["point_estimate", "ci_low", "ci_high"]:
        checks.add(f"business case {key} | {r.unit}", bc[r.unit][key], r[key], rtol=1e-8)

## Summary

In [21]:
checks.report()

55 checks, 55 passed, 0 failed

Largest differences:
                                             check   hexkit        R    abs_diff               tolerance
             fit_gamma shape k (vs MASS::fitdistr)  1.76067  1.76265   0.0019792  atol=1e-08, rtol=0.005
                             fit_gamma scale theta  40.8391  40.8394 0.000295563 atol=1e-08, rtol=0.0001
                                  fit_negbin alpha 0.600603 0.600637 3.38204e-05  atol=1e-08, rtol=0.001
                                   fit_negbin mean    2.413  2.41301 1.49679e-05 atol=1e-08, rtol=0.0001
NB dispersion   counts, 2 groups | per_transaction 0.174781 0.174771 1.01233e-05  atol=1e-08, rtol=0.001


,check,hexkit,R,abs_diff,tolerance,pass
0,"is_normal normal, equal var (ANOVA)",True,True,NaN,exact,True
1,"is_homogeneous normal, equal var (ANOVA)",True,True,NaN,exact,True
2,"test chosen normal, equal var (ANOVA)",Standard ANOVA,Standard ANOVA,NaN,exact,True
3,"p-value normal, equal var (ANOVA)",0.009942,0.009942,2.923009e-15,"atol=1e-10, rtol=1e-06",True
4,"is_normal normal, unequal var (Welch)",False,False,NaN,exact,True
5,"is_homogeneous normal, unequal var (Welch)",False,False,NaN,exact,True
6,"test chosen normal, unequal var (Welch)",Kruskal-Wallis,Kruskal-Wallis,NaN,exact,True
7,"p-value normal, unequal var (Welch)",0.977833,0.977833,0.000000e+00,"atol=1e-10, rtol=1e-06",True
8,"is_normal skewed, 2 groups (Mann-Whitney)",False,False,NaN,exact,True
9,"is_homogeneous skewed, 2 groups (Mann-Whitney)",True,True,NaN,exact,True
